# FMR — Real-Model Pipeline (Instance A)

Runs the Faithful Medical Reasoning pipeline on **real** open medical VLMs on a Colab GPU and pushes results back to `master` automatically.

**Set two Colab secrets (🔑 sidebar) first:** `HF_TOKEN` (HF read token) and `GH_TOKEN` (GitHub PAT, `contents: read+write`). Pick a **T4 GPU** runtime, then run cells top to bottom.

**Memory-safe by design:** images are downscaled to ≤512px (medical scans are high-res and blow up the vision encoder), models load in fp16 and are unloaded between stages, and per-candidate activations are freed. Stages push independently, so a failure never discards earlier results. Start with the **smoke run**.

In [ ]:
# 1. GPU + memory config (set alloc conf BEFORE importing torch)
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # reduce fragmentation (per the OOM hint)
import torch, subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout[:600])
assert torch.cuda.is_available(), "Enable a GPU runtime: Runtime -> Change runtime type -> GPU (T4)"
print("CUDA OK:", torch.cuda.get_device_name(0))

In [ ]:
# 2. Tokens from Colab secrets
import os
try:
    from google.colab import userdata
    for k in ("HF_TOKEN", "GH_TOKEN"):
        try: os.environ[k] = userdata.get(k)
        except Exception: print(f"WARNING: secret {k} not set")
except Exception:
    print("Not in Colab; expecting HF_TOKEN/GH_TOKEN already in env")
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")), "| GH_TOKEN set:", bool(os.environ.get("GH_TOKEN")))

In [ ]:
# 3. Clone the repo (master = Instance A) and install
import os
REPO = "https://github.com/Ankit-blip737/fmr-thesis.git"
if not os.path.exists("fmr-thesis"):
    tok = os.environ.get("GH_TOKEN")
    url = REPO.replace("https://", f"https://x-access-token:{tok}@") if tok else REPO
    !git clone -b master {url} fmr-thesis
%cd fmr-thesis
!git pull --no-edit
!pip -q install -e fmr[real] || pip -q install numpy scipy pyyaml matplotlib scikit-learn "transformers>=4.52" datasets accelerate qwen-vl-utils pillow
print("installed")

## 4. SMOKE RUN — do this first (a few minutes, guaranteed headline)

Small VQA-RAD run (80 samples, 3 consistency chains). Produces baselines + the blind-test replication verdict + the conformal gate and **pushes to `master`** — so you have a real result before the bigger runs.

In [ ]:
!python fmr/scripts/run_real.py     --dataset vqa_rad --reasoning medvlm_r1 --non-reasoning qwen25_vl_3b     --max-samples 80 --n-consistency 3 --alpha 0.15 --push

## 5. Fuller runs — scale up once the smoke run succeeded

Each pushes per-stage. If a run still OOMs, add `--max-image-side 448` isn't a CLI flag — instead lower `--max-samples`. VQA-RAD and PathVQA have inline images; the SLAKE cell fetches its image archive first.

In [ ]:
# VQA-RAD (X-ray/CT), larger
!python fmr/scripts/run_real.py     --dataset vqa_rad --reasoning medvlm_r1 --non-reasoning qwen25_vl_3b     --max-samples 300 --n-consistency 5 --alpha 0.15 --push

In [ ]:
# PathVQA (pathology — modality breadth)
!python fmr/scripts/run_real.py     --dataset pathvqa --reasoning medvlm_r1 --non-reasoning qwen25_vl_3b     --max-samples 300 --n-consistency 5 --alpha 0.15 --push

In [ ]:
# SLAKE — fetch images first (mirror is annotations-only), then run with --image-root
import os
os.makedirs("slake_imgs", exist_ok=True)
!huggingface-cli download BoKelvin/SLAKE imgs.zip --repo-type dataset --local-dir slake_imgs 2>/dev/null || echo "auto-download failed; place SLAKE imgs under slake_imgs/ manually"
!cd slake_imgs && (unzip -oq imgs.zip 2>/dev/null || echo "no imgs.zip")
IMG_ROOT = "slake_imgs" if os.path.exists("slake_imgs") else None
!python fmr/scripts/run_real.py     --dataset slake --reasoning medvlm_r1 --non-reasoning qwen25_vl_3b     --max-samples 300 --n-consistency 5 --alpha 0.10 --image-root {IMG_ROOT} --push

## 6. Inspect what landed (also already pushed to `master`)

In [ ]:
import json, glob
from IPython.display import Image, display
for f in sorted(glob.glob("fmr/outputs/real/*/run_status.json")):
    print(f, "->", json.load(open(f))["status"])
for f in sorted(glob.glob("fmr/outputs/real/*/blind_test.json")):
    rep = json.load(open(f)).get("replication", {})
    print(f"
{f}
  HEADLINE:", rep.get("note"), "| drift slope:", round(rep.get("drift_slope", float('nan')), 4))
for f in sorted(glob.glob("fmr/outputs/real/*/fmr_results.json")):
    r = json.load(open(f)); v = r.get("validation", {}); ab = r["abstention"]
    print(f"
{f}
  per-modality:", r.get("per_modality"))
    print("  AUROC:", {k: round(v[k],3) for k in v if k.startswith("auroc_")})
    for g in ("fs","confidence"):
        print(f"  abstain[{g}] cov={ab[g]['test']['coverage']:.2f} err={ab[g]['test']['retained_error']} AURC={ab[g]['aurc']:.3f}")
for f in sorted(glob.glob("fmr/outputs/real/*/figures/fig1_grounding_drift.png")):
    display(Image(f))